### Online Retail II — Churn Labels

Builds the churn label for each customer at each month-end (a "snapshot").

Rule: for a snapshot date, take every customer who has already bought. If they make no purchase in the next 90 days, `churn = 1`, otherwise `churn = 0`.

## 1. Load the cleaned transactions

In [ ]:
import pandas as pd

df = pd.read_parquet('../data/online_retail_ii_cleaned_v2.parquet')

# we only need who bought and when
df = df[['Customer ID', 'InvoiceDate']]

print(df.shape)
print('Date range:', df['InvoiceDate'].min(), 'to', df['InvoiceDate'].max())

(776591, 2)
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


## 2. Settings

The 90-day rule: a snapshot can only be labeled if the full 90 days after it are inside the data.
So the last snapshot is the last month-end on or before `dataset_end_date - 90 days`.

In [2]:
WINDOW = pd.Timedelta(days=90)                      # how long a customer can stay silent before we call it churn
dataset_end_date = pd.Timestamp('2011-12-09')       # last day in the data
cutoff_date = dataset_end_date - WINDOW             # last snapshot date that still has 90 days after it

print('Last labelable date:', cutoff_date.date())

Last labelable date: 2011-09-10


## 3. One row per customer visit

A customer buying 10 items in one order is one visit, not 10.

In [3]:
visits = df.drop_duplicates()
print('Visits:', len(visits))

Visits: 36479


## 4. Snapshot dates (month-ends)

In [ ]:
# Every month-end from the first month up to the cutoff
snapshots = pd.date_range('2009-12-31', cutoff_date, freq='ME') # monthly

print(len(snapshots), 'snapshots:', snapshots.min().date(), 'to', snapshots.max().date())

21 snapshots: 2009-12-31 to 2011-08-31


## 5. Label each snapshot

For each snapshot:
1. **past** = customers who have bought on or before the snapshot date
2. **future** = customers who bought in the 90 days after it
3. a customer in **past** but not in **future** is churned (`churn = 1`)

In [5]:
frames = []

for snapshot in snapshots:
    # Start of the day after the snapshot, so purchases made on the snapshot day count as "past"
    boundary = snapshot + pd.Timedelta(days=1)

    # Customers who have already bought by the snapshot
    past = visits[visits['InvoiceDate'] < boundary]['Customer ID'].unique()

    # Customers who bought in the next 90 days
    in_window = (visits['InvoiceDate'] >= boundary) & (visits['InvoiceDate'] < boundary + WINDOW)
    future = visits[in_window]['Customer ID'].unique()

    # One row per customer for this snapshot
    snap = pd.DataFrame({'Customer ID': past, 'snapshot': snapshot})
    snap['churn'] = (~snap['Customer ID'].isin(future)).astype(int)   # no purchase in window -> 1
    frames.append(snap)

labels = pd.concat(frames, ignore_index=True)
labels.head()

,Customer ID,snapshot,churn
0,13085,2009-12-31,0
1,13078,2009-12-31,0
2,15362,2009-12-31,1
3,18102,2009-12-31,0
4,12682,2009-12-31,0


## 6. Sanity checks

In [6]:
# Size and overall churn rate
print('Rows:', len(labels))
print('Customers:', labels['Customer ID'].nunique())
print(f"Churn rate: {labels['churn'].mean():.1%}")

# The last snapshot's 90-day window must end inside the data
last_window_end = labels['snapshot'].max() + pd.Timedelta(days=1) + WINDOW
print('Last window ends:', last_window_end.date(), '| inside data:', last_window_end <= dataset_end_date)

# No customer should appear twice in the same snapshot
print('Duplicate (customer, snapshot) rows:', labels.duplicated(['Customer ID', 'snapshot']).sum())

Rows: 75255
Customers: 5224
Churn rate: 57.9%
Last window ends: 2011-11-30 | inside data: True
Duplicate (customer, snapshot) rows: 0


In [7]:
# Churn rate per snapshot - should look steady, with no spike at the end
labels.groupby('snapshot')['churn'].agg(rows='size', churn_rate='mean')

,rows,churn_rate
snapshot,,
2009-12-31,951,0.370137
2010-01-31,1319,0.373768
2010-02-28,1694,0.378394
2010-03-31,2135,0.429977
2010-04-30,2429,0.466447
2010-05-31,2684,0.502981
2010-06-30,2951,0.511352
2010-07-31,3136,0.455676
2010-08-31,3299,0.416793


## 7. Save

In [8]:
labels.to_parquet('../data/customer_month_labels.parquet', index=False)
print('Saved data/customer_month_labels.parquet')

Saved data/customer_month_labels.parquet
